In [59]:
import numpy as np
import pandas as pd

# Load Excel file
file_path = '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Distributed Generation/Datasets/dg9bus.xlsx'
excel_file = pd.ExcelFile(file_path)

# Load each sheet into dataframes
transmission_df = pd.read_excel(excel_file, sheet_name='BUS 9 (Transmisi)')
gen_load_df = pd.read_excel(excel_file, sheet_name='BUS 9 (Generator Load)')
trafo_df = pd.read_excel(excel_file, sheet_name='BUS 9 (Trafo)')

# Preprocess Data
bus_data = gen_load_df[['bus', 'kapasitas_generator_MVA', 'kapasitas_load_MW', 'kapasitas_load_MVAR']]
transmission_data = transmission_df[['line', 'from_bus_trans', 'to_bus_trans', 'length_km', 'R0_ohm_km', 'R1_ohm_km', 'induktansi_H_km', 'kapasitansi_C_km']]
trafo_data = trafo_df[['trafo', 'from_bus_trafo', 'to_bus_trafo', 'trafo_MVA']]

# Manually input cap_trans_MVA
cap_trans_MVA = {
    1: 98.3,
    2: 63.1,
    3: 217,
    4: 150,
    5: 217.2,
    6: 45.4
}

# Function to calculate losses considering transmission and transformer data
def calculate_losses_all_sheets(transmission_data, trafo_data, cap_trans_MVA, dg_bus=None, dg_capacity=0):
    # Transmission line losses
    R0 = transmission_data['R0_ohm_km'].values
    R1 = transmission_data['R1_ohm_km'].values
    length = transmission_data['length_km'].values
    L = transmission_data['induktansi_H_km'].values
    C = transmission_data['kapasitansi_C_km'].values
    
    # Frequency (assuming 50 Hz for power systems)
    f = 50  
    # Calculate inductive reactance: X_L = 2 * pi * f * L
    X_L = 2 * np.pi * f * L
    # Calculate capacitive reactance: X_C = 1 / (2 * pi * f * C)
    X_C = 1 / (2 * np.pi * f * C)
    
    # Calculate impedance Z = sqrt(R^2 + (X_L - X_C)^2)
    Z0 = np.sqrt(R0**2 + (X_L - X_C)**2)
    Z1 = np.sqrt(R1**2 + (X_L - X_C)**2)
    
    # Calculate total line losses using both zero-sequence and positive-sequence impedance
    losses_line = np.sum(Z0 * length) + np.sum(Z1 * length)
    
    # Apply transmission capacity constraints
    transmission_data['cap_trans_MVA'] = transmission_data['line'].map(cap_trans_MVA)
    power_flow = np.minimum(dg_capacity * transmission_data['cap_trans_MVA'].values, transmission_data['cap_trans_MVA'].values)
    
    # Transformer losses and effects
    # Simplified transformer losses as 1% of the MVA rating
    trafo_losses = np.sum(trafo_data['trafo_MVA'].values) * 0.01  
    
    # DG placement impact (affects the transmission losses and transformer losses)
    if dg_bus is not None:
        # Impact of DG on transmission
        dg_bus_indices = (transmission_data['from_bus_trans'] == dg_bus) | (transmission_data['to_bus_trans'] == dg_bus)
        losses_after_dg = np.sum(Z0[~dg_bus_indices] * length[~dg_bus_indices]) + \
                          np.sum(Z1[~dg_bus_indices] * length[~dg_bus_indices]) + \
                          np.sum(Z0[dg_bus_indices] * length[dg_bus_indices] * (1 - dg_capacity)) + \
                          np.sum(Z1[dg_bus_indices] * length[dg_bus_indices] * (1 - dg_capacity))
        
        # Impact of DG on transformer losses
        trafo_bus_indices = (trafo_data['from_bus_trafo'] == dg_bus) | (trafo_data['to_bus_trafo'] == dg_bus)
        trafo_losses_after_dg = np.sum(trafo_data['trafo_MVA'][~trafo_bus_indices].values * 0.01) + \
                                np.sum(trafo_data['trafo_MVA'][trafo_bus_indices].values * 0.01 * (1 - dg_capacity))
    else:
        losses_after_dg = losses_line
        trafo_losses_after_dg = trafo_losses

    total_losses_before = losses_line + trafo_losses
    total_losses_after = losses_after_dg + trafo_losses_after_dg

    return total_losses_before, total_losses_after

# Fitness function to incorporate data from all sheets
def fitness_function_with_dg_all_sheets(position, transmission_data, trafo_data, cap_trans_MVA):
    dg_bus = int(position[0])  # First part of the position is the bus index
    dg_capacity = position[1]  # Second part of the position is the DG capacity (0 to 1)
    
    # Calculate losses after placing DG
    _, losses_after = calculate_losses_all_sheets(transmission_data, trafo_data, cap_trans_MVA, dg_bus, dg_capacity)
    return losses_after

# Sine Cosine Algorithm (SCA) for optimizing DG placement and capacity
def sine_cosine_algorithm_for_dg_all_sheets(transmission_data, trafo_data, cap_trans_MVA, num_iterations=100, num_agents=10):
    dim = 2  # Two parameters to optimize: bus location and DG capacity
    
    # Initialize random positions for agents (solutions)
    positions = np.zeros((num_agents, dim))
    positions[:, 0] = np.random.randint(0, bus_data.shape[0], size=num_agents)  # Random bus selection
    positions[:, 1] = np.random.uniform(0, 1, size=num_agents)  # Random DG capacity (between 0 and 1)
    
    # Best position initialization
    best_position = positions[0].copy()
    best_fitness = fitness_function_with_dg_all_sheets(best_position, transmission_data, trafo_data, cap_trans_MVA)
    
    # SCA Parameters
    a = 2  # Control parameter
    
    # Main loop of SCA
    for t in range(num_iterations):
        r1 = a - t * (a / num_iterations)  # Linearly decreasing parameter
        
        for i in range(num_agents):
            for j in range(dim):
                r2 = np.random.uniform(0, 2 * np.pi)
                r3 = np.random.uniform(0, 2)
                r4 = np.random.uniform(0, 1)
                
                if r4 < 0.5:
                    # Update position using sine
                    positions[i, j] = positions[i, j] + r1 * np.sin(r2) * np.abs(r3 * best_position[j] - positions[i, j])
                else:
                    # Update position using cosine
                    positions[i, j] = positions[i, j] + r1 * np.cos(r2) * np.abs(r3 * best_position[j] - positions[i, j])
            
            # Bound the bus location (integer between 0 and number of buses)
            positions[i, 0] = np.clip(positions[i, 0], 0, bus_data.shape[0] - 1).astype(int)
            # Bound the DG capacity (limit in kW)
            positions[i, 1] = np.clip(positions[i, 1], 0, 0.5)  # Limit DG capacity (e.g. 0 to 500 kW)
            
            # Calculate fitness
            fitness = fitness_function_with_dg_all_sheets(positions[i], transmission_data, trafo_data, cap_trans_MVA)
            
            # Update the best solution if found
            if fitness < best_fitness:
                best_fitness = fitness
                best_position = positions[i].copy()
    
    # Return the best found solution and fitness
    return best_position, best_fitness

# Step 1: Calculate pre-optimization losses
pre_optimization_losses, _ = calculate_losses_all_sheets(transmission_data, trafo_data, cap_trans_MVA)

# Convert pre-optimization losses from W to MW
pre_optimization_losses_mw = pre_optimization_losses / 1_000_000

# Step 2: Run SCA to find the optimal DG placement and capacity
best_dg_position, post_optimization_losses = sine_cosine_algorithm_for_dg_all_sheets(transmission_data, trafo_data, cap_trans_MVA)

# Convert post-optimization losses from W to MW
post_optimization_losses_mw = post_optimization_losses / 1_000_000

# Step 3: Calculate the actual optimal DG capacity based on the fraction
# Set max DG capacity in kW (let's assume the maximum DG is 500 kW)
max_dg_capacity_kw = 500  # Maximum DG capacity in kW
optimal_dg_capacity_kW = best_dg_position[1] * max_dg_capacity_kw  # DG capacity in kW

# Step 4: Output the results in kW
print(f"Pre-optimization losses (MW): {pre_optimization_losses_mw}")
print(f"Optimal DG placement at bus: {int(best_dg_position[0])}")
print(f"Optimal DG capacity in kW: {optimal_dg_capacity_kW}")  # Display in kW
print(f"Post-optimization losses (MW): {post_optimization_losses_mw}")

Pre-optimization losses (MW): 488.0739443100552
Optimal DG placement at bus: 6
Optimal DG capacity in kW: 250.0
Post-optimization losses (MW): 392.6454449296616
